# CBRAIN OASIS Data Audit (Step 1 Closure)

This notebook produces reproducible inspection evidence for:
- schema and basic type profiling
- rows, columns, subjects, repeated visits
- target prevalence for `cognitive_impairment = (CDR > 0)`
- missingness and duplicate checks
- subject-level split leakage check (no subject overlap)


In [ ]:
from __future__ import annotations

import csv
import hashlib
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint


In [ ]:
# Locate CSV robustly for notebook and repo-root execution.
candidates = [
    Path("../data/oasis_dementia_mri_df.csv"),
    Path("data/oasis_dementia_mri_df.csv"),
    Path("/Users/pranjal/Projects/sysBio/projects/cbrain_oasis_cognition/data/oasis_dementia_mri_df.csv"),
]

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate oasis_dementia_mri_df.csv")

with csv_path.open(newline="") as f:
    rows = list(csv.DictReader(f))

columns = list(rows[0].keys())
print("csv_path:", csv_path)
print("rows:", len(rows))
print("columns:", len(columns))
print(columns)


In [ ]:
# Basic type profiling and missingness.
values_by_col = {c: [] for c in columns}
missing_counts = {c: 0 for c in columns}

for row in rows:
    for c in columns:
        val = (row[c] or "").strip()
        if val == "":
            missing_counts[c] += 1
        else:
            values_by_col[c].append(val)

def infer_type(values: list[str]) -> str:
    if not values:
        return "unknown"
    try:
        for v in values:
            int(v)
        return "int"
    except ValueError:
        pass
    try:
        for v in values:
            float(v)
        return "float"
    except ValueError:
        pass
    return "string"

type_profile = {
    c: {
        "inferred_type": infer_type(values_by_col[c]),
        "missing": missing_counts[c],
        "missing_pct": round((missing_counts[c] / len(rows)) * 100, 2),
    }
    for c in columns
}

print("type profile")
pprint(type_profile)


In [ ]:
# Cohort structure, repeated visits, target prevalence, and duplicate checks.
subject_counts = Counter(row["Subject.ID"] for row in rows)
visit_count_distribution = dict(sorted(Counter(subject_counts.values()).items()))
repeated_subjects = sum(1 for _, n in subject_counts.items() if n > 1)

target = []
for row in rows:
    cdr = (row["CDR"] or "").strip()
    if cdr == "":
        continue
    target.append(1 if float(cdr) > 0 else 0)

target_positive = sum(target)
target_negative = len(target) - target_positive
target_positive_rate = round(target_positive / len(target), 4)

rownames_dup_keys = sum(1 for _, n in Counter(r["rownames"] for r in rows).items() if n > 1)
mri_id_dup_keys = sum(1 for _, n in Counter(r["MRI.ID"] for r in rows).items() if n > 1)
subject_visit_dup_keys = sum(
    1 for _, n in Counter((r["Subject.ID"], r["Visit"]) for r in rows).items() if n > 1
)

summary = {
    "subjects": len(subject_counts),
    "repeated_subjects": repeated_subjects,
    "max_visits_per_subject": max(subject_counts.values()),
    "visit_count_distribution": visit_count_distribution,
    "target_total": len(target),
    "target_positive": target_positive,
    "target_negative": target_negative,
    "target_positive_rate": target_positive_rate,
    "duplicate_rownames_keys": rownames_dup_keys,
    "duplicate_mri_id_keys": mri_id_dup_keys,
    "duplicate_subject_visit_keys": subject_visit_dup_keys,
}
pprint(summary)


In [ ]:
# Baseline-only cohort + deterministic subject split leakage check.
by_subject = defaultdict(list)
for row in rows:
    by_subject[row["Subject.ID"]].append(row)

def baseline_key(r: dict[str, str]):
    visit = (r["Visit"] or "").strip()
    mr_delay = (r["MR.Delay"] or "").strip()
    visit_i = int(visit) if visit else 10**9
    delay_i = int(mr_delay) if mr_delay else 10**9
    return (visit_i, delay_i, r["MRI.ID"])

baseline_rows = [sorted(srows, key=baseline_key)[0] for srows in by_subject.values()]
baseline_target_positive = sum(1 for r in baseline_rows if float(r["CDR"]) > 0)

train_subjects = set()
val_subjects = set()
for sid in sorted(by_subject):
    bucket = int(hashlib.sha256(sid.encode()).hexdigest(), 16) % 10
    if bucket < 8:
        train_subjects.add(sid)
    else:
        val_subjects.add(sid)

subject_overlap = train_subjects & val_subjects

split_summary = {
    "baseline_rows": len(baseline_rows),
    "baseline_positive": baseline_target_positive,
    "baseline_negative": len(baseline_rows) - baseline_target_positive,
    "baseline_positive_rate": round(baseline_target_positive / len(baseline_rows), 4),
    "train_subjects": len(train_subjects),
    "validation_subjects": len(val_subjects),
    "subject_overlap_count": len(subject_overlap),
}
pprint(split_summary)
assert len(subject_overlap) == 0, "Leakage detected: subject overlap across train/validation splits"
print("PASS: No subject appears in both train and validation splits.")


## Step 1 decisions locked for Step 2

- Main target remains `cognitive_impairment = (CDR > 0)`.
- Main-model exclusions: `CDR`, `Group`, `MMSE`, `Subject.ID`, `MRI.ID`, `rownames`.
- First-pass cohort policy: baseline-only row per subject.
- Split policy: deterministic subject-level split with zero subject overlap.
